#Mục tiêu
chúng ta sẽ áp dụng quy trình: Bronze (Raw) -> Silver (Cleaned) -> Gold (Processed).

##Cài đặt và Tải dữ liệu (Giai đoạn BRONZE - Dữ liệu gốc)

In [1]:
!pip install kagglehub pyspark

import kagglehub
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, to_timestamp, monotonically_increasing_id

# 1. Khởi tạo Spark
spark = SparkSession.builder.appName("DataVersioning").getOrCreate()

# 2. Tải Dataset từ Kaggle (Amazon Reviews)
path = kagglehub.dataset_download("datafiniti/consumer-reviews-of-amazon-products")
files = [f for f in os.listdir(path) if f.endswith('.csv')]
raw_data_path = os.path.join(path, files[0])

# 3. Đọc dữ liệu gốc (Bronze Layer)
df_bronze = spark.read.csv(raw_data_path, header=True, inferSchema=True)

# Lưu version Bronze (Dữ liệu thô nguyên bản) dưới dạng Parquet để truy xuất nhanh
df_bronze.write.mode("overwrite").parquet("data_v1_bronze.parquet")
print("Đã lưu Version 1: Bronze (Original Data)")

100%|██████████| 16.3M/16.3M [00:00<00:00, 75.9MB/s]

Extracting files...


Đã lưu Version 1: Bronze (Original Data)


##Làm sạch dữ liệu (Giai đoạn SILVER - Version 2)

In [2]:
from pyspark.sql.functions import col, try_to_timestamp, lit, expr

# Đọc lại từ bản Bronze
df_v2 = spark.read.parquet("data_v1_bronze.parquet")

# Định nghĩa các cột có dấu chấm bằng dấu huyền
column_rating = "`reviews.rating`"
column_text = "`reviews.text`"
column_date = "`reviews.date`"

# --- 1. DATA CLEANING ---

# Bước A: Sử dụng expr và try_cast để ép kiểu Rating về Double an toàn
# Những giá trị 'TRUE', 'FALSE' hoặc rác sẽ tự động thành NULL
df_silver = df_v2.withColumn("rating_numeric", expr(f"try_cast({column_rating} AS DOUBLE)"))

# Bước B: Loại bỏ các bản ghi thiếu thông tin (xóa luôn những dòng có rating rác vừa thành NULL)
df_silver = df_silver.dropna(subset=["rating_numeric", column_text])

# Bước C: Loại bỏ dữ liệu trùng lặp theo ID
df_silver = df_silver.dropDuplicates(["id"])

# Bước D: Lọc bỏ rating không hợp lệ (1-5)
df_silver = df_silver.filter((col("rating_numeric") >= 1) & (col("rating_numeric") <= 5))

# Bước E: Xử lý ngày tháng an toàn
date_format = "yyyy-MM-dd'T'HH:mm:ss.SSSX"
df_silver = df_silver.withColumn(
    "review_date_converted",
    try_to_timestamp(col(column_date), lit(date_format))
)

# --- 2. CHUẨN BỊ OUTPUT CHO SILVER ---
# Thay thế cột cũ bằng cột số đã làm sạch để các bước sau không bị lỗi định dạng
df_silver = df_silver.withColumn("reviews_rating_clean", col("rating_numeric"))

# Chọn các cột cần thiết để lưu (Loại bỏ các cột trung gian hoặc cột lỗi cũ nếu muốn)
# Ở đây tôi giữ lại cấu trúc cũ nhưng dùng cột đã cast thành công
df_silver_final = df_silver.drop("rating_numeric")

# Lưu dữ liệu xuống version 2
df_silver_final.write.mode("overwrite").parquet("data_v2_silver.parquet")

print("Đã lưu Version 2: Silver (Cleaned Data) thành công với try_cast!")

Đã lưu Version 2: Silver (Cleaned Data) thành công với try_cast!


##Chuẩn hóa dữ liệu (Giai đoạn GOLD - Version 3)

In [3]:
from pyspark.sql.functions import col, lower, regexp_replace, monotonically_increasing_id

# 1. Đọc lại từ bản Silver (Dữ liệu đã sạch ở Bước 2)
df_v3 = spark.read.parquet("data_v2_silver.parquet")

# 2. Định nghĩa tên cột có dấu chấm bằng dấu huyền để tránh lỗi Unresolved Column
column_text_raw = "`reviews.text`"
column_rating_clean = "`reviews.rating`" # Hoặc reviews_rating_clean tùy bước 2 bạn đặt tên

# --- DATA NORMALIZATION ---

# 1. Chuẩn hóa Text: Viết thường và loại bỏ ký tự đặc biệt
# Sử dụng dấu huyền trực tiếp trong col()
df_gold = df_v3.withColumn("review_content_final", lower(col(column_text_raw)))
df_gold = df_gold.withColumn("review_content_final", regexp_replace(col("review_content_final"), "[^a-z0-9\\s]", ""))

# 2. Min-Max Scaling cho cột Rating (Đưa về khoảng 0-1)
# Công thức: (x - min) / (max - min). Rating từ 1-5 nên sẽ là (x-1)/4
df_gold = df_gold.withColumn("rating_normalized", (col(column_rating_clean) - 1) / 4)

# 3. Thêm cột Index duy nhất cho mỗi dòng
df_gold = df_gold.withColumn("row_index", monotonically_increasing_id())

# --- LƯU VERSION GOLD ---
# Chọn lọc các cột quan trọng để bộ dataset gọn nhẹ, sẵn sàng cho các bước sau
df_final_output = df_gold.select(
    "row_index",
    "id",
    column_rating_clean,
    "rating_normalized",
    "review_content_final",
    "review_date_converted"
)

df_final_output.write.mode("overwrite").parquet("data_v3_gold_final.parquet")

print("Đã lưu Version 3: Gold (Normalized & Ready) thành công!")

Đã lưu Version 3: Gold (Normalized & Ready) thành công!


##Kết quả cuối cùng (GOLD LAYER)

In [4]:
spark.read.parquet("data_v3_gold_final.parquet").show(10)

+---------+--------------------+--------------+-----------------+--------------------+---------------------+
|row_index|                  id|reviews.rating|rating_normalized|review_content_final|review_date_converted|
+---------+--------------------+--------------+-----------------+--------------------+---------------------+
|        0|AV-ETMhgYSSHbkXwpNb9|             5|              1.0|the best kindle e...|  2017-11-10 00:00:00|
|        1|AV-EVZITKZqtpbFMSoqc|             5|              1.0|love my new kindl...|  2018-04-20 00:00:00|
|        2|AV2ElNnuvKc47QAVouhY|             4|             0.75|it slippes around...|  2018-01-14 00:00:00|
|        3|AVpe5Q3sLJeJML43xt5X|             5|              1.0|its great study e...|  2017-07-03 00:00:00|
|        4|AVpe6nyKLJeJML43yOe2|             3|              0.5|better than nothi...|                 NULL|
|        5|AVpe7JuRilAPnD_xQ_M6|             1|              0.0|is amazon kidding...|  2013-10-10 00:00:00|
|        6|AVpe7nGV

#DATA CLEANING

##Kiểm tra dữ liệu thiếu (NULL, NaN)

In [5]:
from pyspark.sql.functions import col, sum

df_bronze.select([
    sum(col(f"`{c}`").isNull().cast("int")).alias(c)
    for c in df_bronze.columns
]).show()

+---+---------+-----------+----+-----+-----+----------+-----------------+---------+----+------------+------------------+------------+----------------+-------------------+-------------------+----------+------------------+--------------+------------------+------------+-------------+----------------+----------+
| id|dateAdded|dateUpdated|name|asins|brand|categories|primaryCategories|imageURLs|keys|manufacturer|manufacturerNumber|reviews.date|reviews.dateSeen|reviews.didPurchase|reviews.doRecommend|reviews.id|reviews.numHelpful|reviews.rating|reviews.sourceURLs|reviews.text|reviews.title|reviews.username|sourceURLs|
+---+---------+-----------+----+-----+-----+----------+-----------------+---------+----+------------+------------------+------------+----------------+-------------------+-------------------+----------+------------------+--------------+------------------+------------+-------------+----------------+----------+
|  0|        0|          0|   0|    0|    0|         0|               

##Xử lý dữ liệu thiếu

In [6]:
# Rename toàn bộ cột (loại bỏ dấu .)
df_bronze = df_bronze.toDF(*[
    c.replace(".", "_") for c in df_bronze.columns
])

from pyspark.sql.functions import expr

# Tạo rating_numeric
df_bronze = df_bronze.withColumn(
    "rating_numeric",
    expr("try_cast(reviews_rating AS DOUBLE)")
)

# Drop NULL
df_clean = df_bronze.dropna(
    subset=["rating_numeric", "reviews_text"]
)

##Xoá dữ liệu trùng (duplicate)

In [7]:
# Đếm số dòng trước
before = df_clean.count()
print("Before:", before)

Before: 28332


In [8]:
# Xóa duplicate
df_clean = df_clean.dropDuplicates(["id"])

In [9]:
# Đếm lại sau khi xoá
after = df_clean.count()
print("After:", after)
print("Removed:", before - after)

After: 65
Removed: 28267


##Loại bỏ dữ liệu sai

###Giá trị âm bất hợp lý

In [10]:
from pyspark.sql.functions import col

df_clean = df_clean.filter(
    (col("rating_numeric") >= 1) & (col("rating_numeric") <= 5)
)

# Nếu có cột số khác (ví dụ numHelpful)
df_clean = df_clean.filter(
    col("reviews_numHelpful") >= 0
)

###Dữ liệu lỗi format

In [11]:
# Rating dạng sai (TRUE/FALSE, string)
from pyspark.sql.functions import expr

df_clean = df_clean.withColumn(
    "rating_numeric",
    expr("try_cast(reviews_rating AS DOUBLE)")
)

# Date lỗi format
from pyspark.sql.functions import try_to_timestamp, lit

df_clean = df_clean.withColumn(
    "review_date_clean",
    try_to_timestamp(col("reviews_date"), lit("yyyy-MM-dd'T'HH:mm:ss.SSSX"))
)

# Loại bỏ các dòng date lỗi
df_clean = df_clean.filter(
    col("review_date_clean").isNotNull()
)

## Chuẩn hoá kiểu dữ liệu

###Chuyển String → Number

In [12]:
from pyspark.sql.functions import expr

df_clean = df_clean.withColumn(
    "rating_numeric",
    expr("try_cast(reviews_rating AS DOUBLE)")
)


In [13]:
# Kiểm tra kiểu dữ liệu
df_clean.printSchema()

root
 |-- id: string (nullable = true)
 |-- dateAdded: timestamp (nullable = true)
 |-- dateUpdated: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- asins: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- primaryCategories: string (nullable = true)
 |-- imageURLs: string (nullable = true)
 |-- keys: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- manufacturerNumber: string (nullable = true)
 |-- reviews_date: string (nullable = true)
 |-- reviews_dateSeen: string (nullable = true)
 |-- reviews_didPurchase: string (nullable = true)
 |-- reviews_doRecommend: boolean (nullable = true)
 |-- reviews_id: string (nullable = true)
 |-- reviews_numHelpful: integer (nullable = true)
 |-- reviews_rating: integer (nullable = true)
 |-- reviews_sourceURLs: string (nullable = true)
 |-- reviews_text: string (nullable = true)
 |-- reviews_title: string (nullable = true)
 |-- reviews_username: st

###Chuyển Date → Datetime

In [14]:
from pyspark.sql.functions import try_to_timestamp, col, lit

df_clean = df_clean.withColumn(
    "review_date_clean",
    try_to_timestamp(
        col("reviews_date"),
        lit("yyyy-MM-dd'T'HH:mm:ss.SSSX")
    )
)

# Loại bỏ dữ liệu lỗi format
df_clean = df_clean.filter(
    col("review_date_clean").isNotNull()
)

In [15]:
# Kiểm tra lại dữ liệu
print("\n===== SAMPLE DATA =====")
df_clean.show(10)


===== SAMPLE DATA =====
+--------------------+-------------------+-------------------+--------------------+----------+------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+-------------------+-------------------+----------+------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------+-------------------+
|                  id|          dateAdded|        dateUpdated|                name|     asins| brand|          categories|   primaryCategories|           imageURLs|                keys|        manufacturer|manufacturerNumber|        reviews_date|    reviews_dateSeen|reviews_didPurchase|reviews_doRecommend|reviews_id|reviews_numHelpful|reviews_rating|  reviews_sourceURLs|        reviews_text|       reviews_title|    reviews_username|          sourceURLs|rating_numeric|  review_d

# PHẦN TIẾP THEO: HOÀN THIỆN TASK 1 THEO HƯỚNG PHỤC VỤ SVD

Các cell phía trên đã làm phần khởi đầu của **Data Cleaning**.  
Phần dưới đây  bổ sung các bước còn thiếu để:

- hoàn thiện **Task 1: Data Cleaning & Normalization**
- tạo **feature matrix** phù hợp cho đề tài **Dimensionality Reduction: SVD**
- chuẩn bị dữ liệu đầu vào cho ** : Truncated SVD**

In [16]:
# Kiểm tra nhanh dữ liệu sạch sau phần
print("===== SCHEMA SAU DATA CLEANING =====")
df_clean.printSchema()

print("===== SỐ DÒNG SAU CLEANING =====")
print(df_clean.count())

print("===== SAMPLE SAU CLEANING =====")
df_clean.select(
    "id",
    "reviews_text",
    "rating_numeric",
    "review_date_clean",
    "reviews_numHelpful"
).show(10, truncate=False)

===== SCHEMA SAU DATA CLEANING =====
root
 |-- id: string (nullable = true)
 |-- dateAdded: timestamp (nullable = true)
 |-- dateUpdated: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- asins: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- primaryCategories: string (nullable = true)
 |-- imageURLs: string (nullable = true)
 |-- keys: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- manufacturerNumber: string (nullable = true)
 |-- reviews_date: string (nullable = true)
 |-- reviews_dateSeen: string (nullable = true)
 |-- reviews_didPurchase: string (nullable = true)
 |-- reviews_doRecommend: boolean (nullable = true)
 |-- reviews_id: string (nullable = true)
 |-- reviews_numHelpful: integer (nullable = true)
 |-- reviews_rating: integer (nullable = true)
 |-- reviews_sourceURLs: string (nullable = true)
 |-- reviews_text: string (nullable = true)
 |-- reviews_title: string (nulla

## Chuẩn hoá dữ liệu cho Task 1

Ở dataset Amazon Product Reviews, mục tiêu cuối không phải chỉ làm sạch bảng dữ liệu, mà là tạo ra dữ liệu đủ tốt để xây dựng **ma trận đặc trưng** cho SVD.

Vì vậy phần normalization sẽ ưu tiên:

- chuẩn hoá **review text**
- chuẩn hoá các cột số sẽ dùng kèm text
- tách đặc trưng thời gian
- lưu riêng một **version sẵn sàng cho SVD**

In [17]:
from pyspark.sql.functions import (
    col, lower, regexp_replace, trim, when,
    year, month, length, coalesce, lit
)

# Bản sao tiếp theo từ df_clean để không phá biến gốc
df_task1 = df_clean

# 1) Chuẩn hoá text:
# - viết thường
# - bỏ ký tự đặc biệt
# - gom khoảng trắng
# - trim đầu cuối
df_task1 = df_task1.withColumn("review_text_clean", lower(col("reviews_text")))
df_task1 = df_task1.withColumn("review_text_clean", regexp_replace(col("review_text_clean"), "[^a-z0-9\s]", " "))
df_task1 = df_task1.withColumn("review_text_clean", regexp_replace(col("review_text_clean"), "\s+", " "))
df_task1 = df_task1.withColumn("review_text_clean", trim(col("review_text_clean")))

# 2) Chuẩn hoá cột số:
# reviews_numHelpful có thể null => thay bằng 0
df_task1 = df_task1.withColumn(
    "reviews_numHelpful_clean",
    coalesce(col("reviews_numHelpful").cast("double"), lit(0.0))
)

# rating_numeric đã được ép kiểu từ phần trước => scale về 0-1
df_task1 = df_task1.withColumn(
    "rating_normalized",
    (col("rating_numeric") - lit(1.0)) / lit(4.0)
)

# 3) Tạo thêm đặc trưng thời gian từ review_date_clean
df_task1 = df_task1.withColumn("review_year", year(col("review_date_clean")))
df_task1 = df_task1.withColumn("review_month", month(col("review_date_clean")))

# 4) Độ dài review - một đặc trưng số đơn giản nhưng hữu ích
df_task1 = df_task1.withColumn("review_length_chars", length(col("review_text_clean")))

# 5) Loại các dòng text rỗng sau chuẩn hoá
df_task1 = df_task1.filter(
    col("review_text_clean").isNotNull() & (col("review_text_clean") != "")
)

print("===== SỐ DÒNG SAU NORMALIZATION =====")
print(df_task1.count())

df_task1.select(
    "id",
    "review_text_clean",
    "rating_numeric",
    "rating_normalized",
    "reviews_numHelpful_clean",
    "review_year",
    "review_month",
    "review_length_chars"
).show(10, truncate=False)

<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:16: SyntaxWarning: invalid escape sequence '\s'
<>:15: SyntaxWarning: invalid escape sequence '\s'
<>:16: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_3562/3093342495.py:15: SyntaxWarning: invalid escape sequence '\s'
  df_task1 = df_task1.withColumn("review_text_clean", regexp_replace(col("review_text_clean"), "[^a-z0-9\s]", " "))
/tmp/ipykernel_3562/3093342495.py:16: SyntaxWarning: invalid escape sequence '\s'
  df_task1 = df_task1.withColumn("review_text_clean", regexp_replace(col("review_text_clean"), "\s+", " "))


===== SỐ DÒNG SAU NORMALIZATION =====
38
+--------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+-----------------+------------------------+-----------+------------+-------------------+
|id                  |review_text_clean                                                                                                     

In [18]:
# Lưu version hoàn chỉnh của Task 1
task1_output_path = "data_v4_task1_ready_for_svd.parquet"

df_task1.write.mode("overwrite").parquet(task1_output_path)
print(f"Đã lưu version Task 1 hoàn chỉnh: {task1_output_path}")

Đã lưu version Task 1 hoàn chỉnh: data_v4_task1_ready_for_svd.parquet


In [19]:
# Kiểm tra lại dữ liệu sau Task 1
from pyspark.sql.functions import sum

print("===== KIỂM TRA NULL Ở CÁC CỘT CHÍNH SAU TASK 1 =====")
df_task1.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in [
        "id",
        "review_text_clean",
        "rating_numeric",
        "rating_normalized",
        "reviews_numHelpful_clean",
        "review_date_clean",
        "review_year",
        "review_month",
        "review_length_chars"
    ]
]).show()

print("===== SAMPLE OUTPUT TASK 1 =====")
df_task1.select(
    "id",
    "review_text_clean",
    "rating_numeric",
    "rating_normalized",
    "reviews_numHelpful_clean",
    "review_year",
    "review_month"
).show(10, truncate=False)

===== KIỂM TRA NULL Ở CÁC CỘT CHÍNH SAU TASK 1 =====
+---+-----------------+--------------+-----------------+------------------------+-----------------+-----------+------------+-------------------+
| id|review_text_clean|rating_numeric|rating_normalized|reviews_numHelpful_clean|review_date_clean|review_year|review_month|review_length_chars|
+---+-----------------+--------------+-----------------+------------------------+-----------------+-----------+------------+-------------------+
|  0|                0|             0|                0|                       0|                0|          0|           0|                  0|
+---+-----------------+--------------+-----------------+------------------------+-----------------+-----------+------------+-------------------+

===== SAMPLE OUTPUT TASK 1 =====
+--------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------

# TASK  DIMENSIONALITY REDUCTION USING SVD

Với Amazon Product Reviews, cách làm đúng chất đề tài là:

1. dùng **review text** đã chuẩn hoá để tạo **TF-IDF matrix**
2. ghép thêm một số đặc trưng số như:
   - `rating_normalized`
   - `reviews_numHelpful_clean`
   - `review_year`
   - `review_month`
   - `review_length_chars`
3. áp dụng **Truncated SVD** để giảm chiều

Lý do dùng **Truncated SVD**:
- TF-IDF tạo ra ma trận rất lớn và rất thưa (sparse matrix)
- Truncated SVD phù hợp hơn PCA trong tình huống này

In [20]:
# Cài thư viện cần cho Task 2 nếu môi trường chưa có
!pip install -q scikit-learn scipy pandas pyarrow

In [21]:
# Chuyển dữ liệu từ Spark sang pandas cho bước TF-IDF + Truncated SVD
# Có thể giới hạn sample để tránh tràn RAM trên Colab
SVD_SAMPLE_SIZE = 50000

pdf_task2 = (
    df_task1
    .select(
        "id",
        "review_text_clean",
        "rating_normalized",
        "reviews_numHelpful_clean",
        "review_year",
        "review_month",
        "review_length_chars"
    )
    .limit(SVD_SAMPLE_SIZE)
    .toPandas()
)

print("Shape pandas dùng cho SVD:", pdf_task2.shape)
pdf_task2.head()

Shape pandas dùng cho SVD: (38, 7)


,id,review_text_clean,rating_normalized,reviews_numHelpful_clean,review_year,review_month,review_length_chars
0,AV-ETMhgYSSHbkXwpNb9,the best kindle ever for me is still the huge ...,1.0,3.0,2017,11,692
1,AV-EVZITKZqtpbFMSoqc,love my new kindle i would recommend this to a...,1.0,0.0,2018,4,83
2,AVpe7JuRilAPnD_xQ_M6,is amazon kidding me they want me to pay 19 99...,0.0,621.0,2013,10,160
3,AVpf2cQm1cnluZ0-sb5y,since the details for the items are a little s...,1.0,288.0,2009,2,257
4,AVpfIfGA1cnluZ0-emyp,it seems to work just like any other usb plug ...,1.0,0.0,2017,1,56


In [22]:
# Làm sạch lần cuối ở mức pandas để đảm bảo đầu vào cho sklearn không lỗi
import pandas as pd
import numpy as np

pdf_task2["review_text_clean"] = pdf_task2["review_text_clean"].fillna("").astype(str)

numeric_cols = [
    "rating_normalized",
    "reviews_numHelpful_clean",
    "review_year",
    "review_month",
    "review_length_chars"
]

for c in numeric_cols:
    pdf_task2[c] = pd.to_numeric(pdf_task2[c], errors="coerce").fillna(0.0)

print(pdf_task2[numeric_cols].isna().sum())
print((pdf_task2["review_text_clean"] == "").sum(), "dòng có text rỗng")

rating_normalized           0
reviews_numHelpful_clean    0
review_year                 0
review_month                0
review_length_chars         0
dtype: int64
0 dòng có text rỗng


In [23]:
# Tạo TF-IDF matrix từ review text
from sklearn.feature_extraction.text import TfidfVectorizer

MAX_FEATURES = 5000

tfidf = TfidfVectorizer(
    max_features=MAX_FEATURES,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

X_text = tfidf.fit_transform(pdf_task2["review_text_clean"])

print("TF-IDF matrix shape:", X_text.shape)

TF-IDF matrix shape: (38, 18)


In [24]:
# Ghép thêm các đặc trưng số vào ma trận đặc trưng
from scipy.sparse import csr_matrix, hstack

X_num = csr_matrix(pdf_task2[numeric_cols].values)
X = hstack([X_text, X_num])

print("Feature matrix cuối cùng shape:", X.shape)

Feature matrix cuối cùng shape: (38, 23)


In [26]:
# Áp dụng Truncated SVD
from sklearn.decomposition import TruncatedSVD

max_allowed_components = min(X.shape[0] - 1, X.shape[1] - 1)

# số chiều mong muốn ban đầu
TARGET_COMPONENTS = 100

# tự động điều chỉnh để không vượt quá số chiều thực tế
N_COMPONENTS = min(TARGET_COMPONENTS, max_allowed_components)

if N_COMPONENTS < 2:
    raise ValueError(
        f"Không thể chạy TruncatedSVD vì số chiều quá thấp. "
        f"X.shape={X.shape}, N_COMPONENTS tính được={N_COMPONENTS}"
    )

print("Shape trước giảm chiều:", X.shape)
print("TARGET_COMPONENTS:", TARGET_COMPONENTS)
print("N_COMPONENTS thực tế dùng:", N_COMPONENTS)

svd_model = TruncatedSVD(
    n_components=N_COMPONENTS,
    n_iter=10,
    random_state=42
)

X_reduced = svd_model.fit_transform(X)

explained_variance_ratio = svd_model.explained_variance_ratio_
cumulative_explained_variance = explained_variance_ratio.cumsum()

print("Shape sau giảm chiều:", X_reduced.shape)
print("Tổng explained variance ratio:", explained_variance_ratio.sum())

Shape trước giảm chiều: (38, 23)
TARGET_COMPONENTS: 100
N_COMPONENTS thực tế dùng: 22
Shape sau giảm chiều: (38, 22)
Tổng explained variance ratio: 1.0000000000000004


In [27]:
# Tạo bảng explained variance để phân tích
svd_report = pd.DataFrame({
    "component": np.arange(1, len(explained_variance_ratio) + 1),
    "explained_variance_ratio": explained_variance_ratio,
    "cumulative_explained_variance": cumulative_explained_variance
})

In [28]:
# Lưu output của Task 2
actual_components = X_reduced.shape[1]
reduced_cols = [f"svd_component_{i}" for i in range(1, actual_components + 1)]

reduced_df = pd.DataFrame(X_reduced, columns=reduced_cols)
reduced_df.insert(0, "id", pdf_task2["id"].values)

reduced_output_csv = "data_v5_svd_reduced.csv"
report_output_csv = "data_v5_svd_explained_variance.csv"

reduced_df.to_csv(reduced_output_csv, index=False)
svd_report.to_csv(report_output_csv, index=False)

print(f"Đã lưu dữ liệu sau giảm chiều: {reduced_output_csv}")
print(f"Đã lưu báo cáo explained variance: {report_output_csv}")

Đã lưu dữ liệu sau giảm chiều: data_v5_svd_reduced.csv
Đã lưu báo cáo explained variance: data_v5_svd_explained_variance.csv


# Kết luận ngắn cho bài



- **Task 1** đã hoàn thiện theo hướng phục vụ SVD:
  - làm sạch dữ liệu
  - chuẩn hoá review text
  - chuẩn hoá rating và numHelpful
  - tạo đặc trưng thời gian
  - lưu version sẵn sàng cho mô hình

-  đã xây dựng đúng pipeline của đề tài:
  - text → TF-IDF
  - ghép numeric features
  - giảm chiều bằng **Truncated SVD**
  - lưu dữ liệu sau giảm chiều và báo cáo explained variance

Như vậy notebook này đã đi đúng logic của đề tài **Dimensionality Reduction: SVD** với **Amazon Product Reviews Dataset**.